
# HAM10000 External Domain Generalization Full Pipeline

Complete experiment pipeline:

1. Load HAM10000 external dataset.
2. Prepare lesion-level train/validation/test split.
3. Load ISIC2019-trained checkpoints.
4. Zero-shot evaluation:
   - Supervised ResNet50
   - Rotation SSL
   - SimCLR No-TTA
   - SimCLR TTA
5. Feature fusion evaluation:
   - FusionConcat
   - Gated Fusion
   - Residual Fusion
6. Ensemble evaluation:
   - Weighted ensemble
   - Stacking
   - Class-bias corrected final model
7. HAM10000 domain adaptation:
   - Fine tune
   - Validation selection
   - Test evaluation
8. Bootstrap confidence intervals.
9. Grad-CAM/XAI export.

Checkpoint paths are configurable in the CONFIG cell.


In [1]:

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"

OUTPUT=Path("/kaggle/working/HAM10000_EXTERNAL_RESULTS")
OUTPUT.mkdir(exist_ok=True)

print(DEVICE)


cuda


In [2]:

# ==============================
# CONFIGURATION
# ==============================

CONFIG = {

"HAM_ROOT": "/kaggle/input",

"IMAGE_SIZE":224,

"NUM_CLASSES":8,

"CHECKPOINTS":{

"supervised":"",
"rotation":"",
"simclr":"",
"fusion":"",
"stacking":""

}

}

print(CONFIG)


{'HAM_ROOT': '/kaggle/input', 'IMAGE_SIZE': 224, 'NUM_CLASSES': 8, 'CHECKPOINTS': {'supervised': '', 'rotation': '', 'simclr': '', 'fusion': '', 'stacking': ''}}


## 1. Automatic HAM10000 loading

In [3]:

HAM_META=None

for p in Path(CONFIG["HAM_ROOT"]).rglob("*"):
    if p.is_file() and "metadata" in p.name.lower():
        HAM_META=p
        break

if HAM_META is None:
    raise FileNotFoundError("HAM metadata not found")

HAM=pd.read_csv(HAM_META)

image_map={}

for p in Path(CONFIG["HAM_ROOT"]).rglob("*.jpg"):
    image_map[p.stem]=str(p)

HAM["path"]=HAM["image_id"].map(image_map)

HAM=HAM.dropna(subset=["path"])

print(HAM.shape)
HAM.head()


(10015, 9)


     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset                                               path  
0  vidir_modern  /kaggle/input/datasets/anirbanroysudipto/skin-...  
1  vidir_modern  /kaggle/input/datasets/anirbanroysudipto/skin-...  
2  vidir_modern  /kaggle/input/datasets/anirbanroysudipto/skin-...  
3  vidir_modern  /kaggle/input/datasets/anirbanroysudipto/skin-...  
4  vidir_modern  /kaggle/input/datasets/anirbanroysudipto/skin-...  

## 2. HAM class preparation

In [4]:

CLASS_MAP={
"mel":0,
"nv":1,
"bcc":2,
"akiec":3,
"bkl":4,
"df":5,
"vasc":6
}

HAM["label"]=HAM["dx"].map(CLASS_MAP)

HAM=HAM.dropna(subset=["label"])

HAM["label"]=HAM["label"].astype(int)

print(HAM.label.value_counts())


label
1    6705
0    1113
4    1099
2     514
3     327
6     142
5     115
Name: count, dtype: int64


## 3. Lesion-level split

In [5]:

from sklearn.model_selection import GroupShuffleSplit

g1=GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx,temp_idx=next(
    g1.split(HAM,groups=HAM["lesion_id"])
)

HAM_TRAIN=HAM.iloc[train_idx]
TEMP=HAM.iloc[temp_idx]

g2=GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx,test_idx=next(
    g2.split(TEMP,groups=TEMP["lesion_id"])
)

HAM_VAL=TEMP.iloc[val_idx]
HAM_TEST=TEMP.iloc[test_idx]

print(len(HAM_TRAIN),len(HAM_VAL),len(HAM_TEST))


7002 1519 1494


## 4. Dataset and dataloaders

In [6]:

from torchvision import transforms
from torch.utils.data import Dataset,DataLoader

transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


class HAMDataset(Dataset):

    def __init__(self,df,transform):
        self.df=df.reset_index(drop=True)
        self.transform=transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        row=self.df.iloc[i]
        img=Image.open(row.path).convert("RGB")
        img=self.transform(img)
        return img,row.label


train_loader=DataLoader(
    HAMDataset(HAM_TRAIN,transform),
    batch_size=32,
    shuffle=True
)

test_loader=DataLoader(
    HAMDataset(HAM_TEST,transform),
    batch_size=32
)


## 5. Model loading section

In [7]:

# Load your saved ISIC2019 models here.
# The architecture must match the original training notebook.

def load_checkpoint(model,path):
    if path and os.path.exists(path):
        state=torch.load(path,map_location=DEVICE)
        model.load_state_dict(state)
    return model


print("Checkpoint loader ready")


Checkpoint loader ready


## 6. Zero-shot evaluation

In [8]:

def evaluate_predictions(y_true,y_pred):

    return {
        "accuracy":accuracy_score(y_true,y_pred),
        "balanced_accuracy":balanced_accuracy_score(y_true,y_pred),
        "macro_f1":f1_score(y_true,y_pred,average="macro")
    }


# Run:
# supervised
# rotation
# simclr
# fusion
# ensemble

# Save:
# zero_shot_results.csv


## 7. Fusion and ensemble evaluation

In [9]:

# Use saved features/probabilities from ISIC2019 experiments.

# Generate:
# fusion_results.csv
# ensemble_results.csv


## 8. HAM domain adaptation fine tuning

In [10]:

criterion=nn.CrossEntropyLoss()


def train_one_epoch(model,loader,optimizer):

    model.train()
    total=0

    for x,y in loader:

        x=x.to(DEVICE)
        y=y.to(DEVICE)

        optimizer.zero_grad()

        out=model(x)

        loss=criterion(out,y)

        loss.backward()

        optimizer.step()

        total+=loss.item()

    return total/len(loader)


# Fine tune:
# ResNet
# SimCLR classifier
# Fusion model

# Save:
# HAM_domain_adaptation_results.csv


## 9. Bootstrap uncertainty

In [11]:

def bootstrap_metric(y_true,y_pred,n=1000):

    scores=[]

    rng=np.random.default_rng(42)

    for _ in range(n):

        idx=rng.choice(
            len(y_true),
            len(y_true),
            replace=True
        )

        scores.append(
            balanced_accuracy_score(
                np.array(y_true)[idx],
                np.array(y_pred)[idx]
            )
        )

    return (
        np.mean(scores),
        np.percentile(scores,2.5),
        np.percentile(scores,97.5)
    )

# Save bootstrap_results.csv


## 10. Grad-CAM/XAI export

In [12]:

# Generate:
# correct prediction heatmaps
# wrong prediction heatmaps

# Save:
# gradcam/
